# 第 6 章 ロジスティック回帰

「0 か 1 か」という硬い予測を、シグモイド関数で「0 から 1 の確率」に置き換えます。

対応する記事: [第 6 章 ロジスティック回帰（Python 版）](https://github.com/k2works/grokking-machine-learning-excersice/blob/main/docs/article/grokking-machine-learning/python/ch06.md)

実装本体: `apps/grokking-ml-python/src/`

## セットアップ

実装本体（`../src/grokking_ml/`）を読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

```bash
cd apps/grokking-ml-python
uv sync
uv run jupyter lab notebooks/
```

In [1]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

from grokking_ml.ch06_logistic_regression import *

## シグモイド関数

実数を 0 から 1 の範囲へ押し込む関数です。**大きな負の入力でも壊れない実装** になっていることを確かめます（素朴な `1/(1+exp(-x))` は Python では例外になります）。

In [2]:
for x in [-1000.0, -5.0, -1.0, 0.0, 1.0, 5.0, 1000.0]:
    print(f"sigmoid({x:>8.1f}) = {sigmoid(x):.6f}")

print()
print("対称性 sigmoid(2) + sigmoid(-2) =", sigmoid(2.0) + sigmoid(-2.0))

sigmoid( -1000.0) = 0.000000
sigmoid(    -5.0) = 0.006693
sigmoid(    -1.0) = 0.268941
sigmoid(     0.0) = 0.500000
sigmoid(     1.0) = 0.731059
sigmoid(     5.0) = 0.993307
sigmoid(  1000.0) = 1.000000

対称性 sigmoid(2) + sigmoid(-2) = 0.9999999999999999


## 対数損失は「確信の度合い」を測る

**当たったかどうかではなく、どれくらいの確信で当たったか** を測ります。0.51 で正解しても損失は 0.67 残るので、学習はまだ進みます。

In [3]:
import math

print(f"{'予測確率':>10} {'正解が 1 のときの損失':>22}")
for probability in [0.99, 0.9, 0.51, 0.5, 0.1, 0.01]:
    print(f"{probability:>10.2f} {-math.log(probability):>22.4f}")

      予測確率           正解が 1 のときの損失
      0.99                 0.0101
      0.90                 0.1054
      0.51                 0.6733
      0.50                 0.6931
      0.10                 2.3026
      0.01                 4.6052


## 学習

第 5 章と同じデータを使います。**初期の損失 0.6931 は `-log(0.5)`**、つまり「すべて五分五分」の状態です。第 5 章のパーセプトロン誤差が初期状態で 0 だったのと対照的です。

In [4]:
points = [(1.0, 0.0), (0.0, 2.0), (1.0, 1.0), (1.0, 2.0),
          (1.0, 3.0), (2.0, 2.0), (2.0, 3.0), (3.0, 2.0)]
labels = [0, 0, 0, 0, 1, 1, 1, 1]

trained, losses = logistic_regression(points, labels, learning_rate=0.1, epochs=1000, seed=0)

print("重み  ", [round(w, 4) for w in trained.weights])
print(f"バイアス {trained.bias:.4f}")
print(f"損失   {losses[0]:.4f} → {losses[-1]:.4f}")
print(f"正解率 {accuracy(trained, points, labels):.2f}")

重み   [2.2506, 1.5327]
バイアス -5.6115
損失   0.6931 → 0.1638
正解率 1.00


## 確率としての出力

単に分類できているだけでなく、**どちらがより確からしいかまで答えられます。**

In [5]:
for point, label in zip(points, labels):
    probability = trained.predict_probability(point)
    bar = "#" * int(probability * 40)
    print(f"({point[0]:.0f},{point[1]:.0f}) 正解={label}  {probability:.4f} {bar}")

(1,0) 正解=0  0.0335 #
(0,2) 正解=0  0.0727 ##
(1,1) 正解=0  0.1385 #####
(1,2) 正解=0  0.4267 #################
(1,3) 正解=1  0.7751 ###############################
(2,2) 正解=1  0.8760 ###################################
(2,3) 正解=1  0.9703 ######################################
(3,2) 正解=1  0.9853 #######################################


## 試してみる: 閾値を変える

**確率が手に入ると、再学習せずに判定の厳しさを変えられます。** 第 7 章で扱う適合率と再現率のトレードオフに直結します。

In [6]:
for threshold in [0.2, 0.5, 0.8]:
    predictions = [trained.predict(p, threshold=threshold) for p in points]
    correct = sum(1 for p, l in zip(predictions, labels) if p == l)
    print(f"閾値 {threshold}  予測 {predictions}  正解数 {correct}/{len(labels)}")

閾値 0.2  予測 [0, 0, 0, 1, 1, 1, 1, 1]  正解数 7/8
閾値 0.5  予測 [0, 0, 0, 0, 1, 1, 1, 1]  正解数 8/8
閾値 0.8  予測 [0, 0, 0, 0, 0, 1, 1, 1]  正解数 7/8
